# VitalRegistro — APK no Google Colab

Execute as células em ordem em um runtime **CPU, Linux x86_64**. O notebook detecta o Python do kernel (o Colab anunciou a migração para 3.13) e prepara **CPython 3.14.7 isolado** para Buildozer/p4a. Não altera o kernel nem exige downgrade.

Combinação: Buildozer `1.6.1.dev0` em commit fixo, p4a `develop` em commit fixo, API 36, NDK 29 e Java 17. O Python embarcado segue a recipe p4a 3.14.2; Kivy 2.3.1. Consulte [ANDROID.md](https://github.com/akioShaolin/vital-registro/blob/main/docs/ANDROID.md).

**Publique os arquivos atualizados no GitHub antes de usar este notebook.** Ele clona o código remoto. Build e instalação em aparelho ainda precisam de validação real; não há garantia de APK apenas por passar nas verificações de ambiente.


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

print('Python do kernel:', sys.version)
print('Executável:', sys.executable)
print('Sistema:', platform.platform(), platform.machine())
if sys.platform != 'linux' or platform.machine() != 'x86_64':
    raise RuntimeError('Selecione um runtime Colab CPU Linux x86_64.')
if sys.version_info < (3, 10):
    raise RuntimeError('O bootstrap requer Python >=3.10; o compilador usa um Python separado.')

REPO_URL = 'https://github.com/akioShaolin/vital-registro.git'
project = Path('/content/vital-registro')
build_succeeded = False
if not project.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(project)], check=True)
else:
    remote = subprocess.run(['git', '-C', str(project), 'remote', 'get-url', 'origin'],
                            check=True, text=True, stdout=subprocess.PIPE).stdout.strip()
    if remote != REPO_URL:
        raise RuntimeError('A pasta existente pertence a outro remoto. Inicie uma sessão limpa.')
    print('Reutilizando o clone existente, sem apagar arquivos nem atualizar seu histórico.')
os.chdir(project)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
if not (project / 'scripts/build_environment.py').is_file():
    raise RuntimeError('O clone ainda está no fluxo antigo. Publique as alterações e use uma sessão limpa.')


## Instalar ferramentas e escolher o Python de compilação

O setup instala Java 17 e dependências Linux, usa uv para baixar CPython 3.14.7 se necessário e cria `.venv-build/`. O kernel permanece intacto. Um ambiente antigo incompatível provoca um erro explicativo, sem exclusão automática. Todos os comandos propagam falhas.


In [ ]:
setup_env = os.environ.copy()
setup_env['VITALREGISTRO_BOOTSTRAP_PYTHON'] = sys.executable
subprocess.run(['bash', 'scripts/setup_linux.sh'], env=setup_env, check=True)
build_python = project / '.venv-build/bin/python'
subprocess.run([str(build_python), '--version'], check=True)
subprocess.run([str(build_python), '-m', 'pip', 'check'], check=True)


## Executar testes no Python usado para compilar

Os testes não precisam da interface gráfica nem de instalar Kivy no kernel. O p4a compila a dependência Android com a recipe apropriada.


In [ ]:
subprocess.run([str(build_python), '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run([str(build_python), '-m', 'compileall', '-q', 'main.py', 'vitalregistro', 'scripts', 'tests'], check=True)


## Licenças do Android SDK

Leia as [condições do Android SDK](https://developer.android.com/studio/terms) e altere a opção abaixo somente se concordar. A configuração é modificada apenas no clone temporário do Colab. O aceite não adiciona permissões ao aplicativo.


In [ ]:
import configparser

ACCEPT_ANDROID_SDK_LICENSES = False
if not ACCEPT_ANDROID_SDK_LICENSES:
    raise RuntimeError('Leia as condições do SDK e defina ACCEPT_ANDROID_SDK_LICENSES=True para continuar.')
spec_path = project / 'buildozer.spec'
spec = configparser.ConfigParser(interpolation=None)
spec.read(spec_path, encoding='utf-8')
spec.set('app', 'android.accept_sdk_license', 'True')
with spec_path.open('w', encoding='utf-8') as output:
    spec.write(output)


## Compilar o APK debug

A saída detalhada permanece visível. Downloads de SDK/NDK e compilação nativa podem consumir vários GB e demorar. O comando interrompe em erro e só libera a etapa de download após sucesso. Não reaproveite caches do fluxo antigo com NDK 25b/p4a master.


In [ ]:
build_succeeded = False
subprocess.run([str(build_python), 'scripts/build_android.py'], check=True)
build_succeeded = True


## Baixar os APKs gerados

Baixe antes de encerrar a sessão. Um APK compilado ainda precisa ser instalado e testado; a execução do build não comprova funcionamento em aparelho.


In [ ]:
if not build_succeeded:
    raise RuntimeError('Execute a compilação com sucesso antes de baixar o APK.')
from google.colab import files

apks = sorted((project / 'bin').glob('*.apk'))
if not apks:
    raise RuntimeError('Nenhum APK em bin/. Confira os logs da compilação.')
for apk in apks:
    print(apk.name, apk.stat().st_size, 'bytes')
    files.download(str(apk))
